# QuantJourney SDK - Public Signals Overlay Intelligence (Reference Basket)

This notebook demonstrates a QuantJourney SDK workflow that overlays public signals across earnings, congress, 13F, macro, COT and volatility for a book-level watchlist.

It covers:

- Direct QuantJourney SDK calls for the required market, macro, regulatory or portfolio data
- Transparent pandas/numpy calculations so research assumptions stay visible
- Chart-ready output that can be reused in notebooks, reports or API documentation

## Prerequisites

Make sure you have:

- Access to QuantJourney API (https://api.quantjourney.cloud)
- `QJ_API_KEY` configured in your environment
- Tenant access to the connectors used by this example

In [ ]:
import os
from typing import Any
from datetime import datetime, timedelta
import pandas as pd
from quantjourney.sdk import QuantJourneyAPI
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2024-01-01')
END = os.getenv('QJ_EXAMPLE_END', '2026-06-06')

def unwrap(payload: Any) -> Any:
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(p):
    v = unwrap(p)
    if isinstance(v, list):
        return v
    if isinstance(v, dict):
        for k in ('rows', 'data', 'items', 'earnings', 'filings', 'trades'):
            if isinstance(v.get(k), list):
                return v[k]
        return [v]
    return []
basket = ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'META', 'GOOGL', 'JPM']
print('=== Public Signals Overlay Intelligence ===\nBasket:', basket)
earnings = qj.fmp.get_earnings_calendar(from_date=(datetime.now() - timedelta(days=45)).date(), to_date=datetime.now().date())
earn_df = pd.DataFrame(as_rows(earnings))
if not earn_df.empty:
    earn_df = earn_df[earn_df.get('symbol', '').isin(basket)]
print('Recent earnings in basket:', len(earn_df))
congress_hits = []
for sym in basket[:4]:
    for src in [qj.fmp.get_house_trades, qj.fmp.get_senate_trades]:
        t = as_rows(src(symbol=sym))
        if t:
            congress_hits.append({'symbol': sym, 'count_recent': len(t)})
print('Congress activity sample:', pd.DataFrame(congress_hits).head())
macro_events = getattr(qj, 'macro', qj).get_economic_events(from_date=str(datetime.now().date() - timedelta(days=10)))
print('Recent macro events (count):', len(as_rows(macro_events)))
vix = qj.cboe.get_vix_data()
if vix is not None:
    print('VIX data available for regime overlay.')
print('\nThis notebook produces the raw signals a human analyst would combine into a one-pager.\nIn a fuller version: rank by impact, attach request_ids, and output a clean markdown/HTML brief.')


## Notes

Example for daily intelligence style output using only public multi-source signals (earnings/estimates, congress, 13F, macro calendar, COT, vol).
Feed a real holdings list (from 13F example or index constituents) to turn it into a book-relevant packet.
No PMS or IBOR data is accessed or required.